## Load Data

In [11]:
"""Load Data
Structure:
    1. Imports, Variables, Functions
    2. Load Data
"""

# 1. Imports, Variables, Functions
# imports
import xml.etree.ElementTree as ET, os
import pandas as pd, numpy as np, os, sys
import anndata as ad
import logging
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import json
from typing import *
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.metrics import confusion_matrix, classification_report

logging.basicConfig(level=logging.INFO)

# variables

run_dir = os.path.join("..", "outputs", "run-24-10-12-01")
# run_dir = os.path.join("..", "outputs", "run-24-09-27-10")
output_dir = os.path.join(run_dir, "outputs")
split_n = 0
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

mesh_file_path = os.path.join(
    "/aloy/home/ddalton/projects/disease_signatures/data/MeSH/desc2023.xml"
)

mesh_da_mappings_path = "/aloy/home/ddalton/projects/disease_signatures/data/DiSignAtlas/mesh_tree_terms.pkl"

df_info_path = os.path.join(
    "/aloy",
    "home",
    "ddalton",
    "projects",
    "disease_signatures",
    "data",
    "DiSignAtlas",
    "Disease_information_Datasets_extended.csv",
)

# functions


def load_run_output(input_dir: str) -> tuple:
    """Load the output of a run
    Args:
        input_dir (str): path to the run output directory
    Returns:
        loaded_variables (tuple): tuple of loaded variables
    """

    variables_to_load = [
        # "split",
        # "predictions_test",
        "labels_test",
        # "results_test",
        # "all_outputs_test",
        # "predictions_train",
        "labels_train",
        # "results_train",
        # "all_outputs_train",
        "adata_orig",
        # "id2type",
    ]

    # initialize loaded variables as an empty tuple
    loaded_variables = ()

    # loop through variables
    for variable in variables_to_load:
        if variable.startswith("adata"):
            if False:
                # load everything
                loaded_variable = ad.read_h5ad(
                    os.path.join(input_dir, f"{variable}.h5ad")
                )

            else:
                # do not load everything
                loaded_variable = ad.read_h5ad(
                    os.path.join(input_dir, f"{variable}.h5ad"), backed="r"
                )

        else:
            with open(os.path.join(input_dir, f"{variable}.pkl"), "rb") as f:
                loaded_variable = pickle.load(f)

        # add the loaded variable to the tuple
        loaded_variables += (loaded_variable,)

    print(f"Nº of loaded variables {len(loaded_variables)}")

    return loaded_variables


def load_comparison_matrix(
    output_dir: str, comparison_type: str, split_type: str
) -> np.matrix:
    """Load the comparison matrix
    Args:
        - output_dir (str): path to the output directory
        - comparison_type (str): type of comparison matrix to load
        - split_type (str): type of split to load
    Returns:
        - matrix (np.matrix): comparison matrix
    """

    if comparison_type == "Cosine Similarity":
        file_name = "c_matrix"
    if comparison_type == "Euclidean Distance":
        file_name = "e_matrix"
    if comparison_type == "Pearson Correlation":
        file_name = "p_matrix"

    file_name = f"{file_name}_{split_type}_parallel.npy"

    matrix = np.load(os.path.join(output_dir, file_name))

    return matrix


def shorten_tree(tree_ids) -> List:
    """For a given list of tree ids return the shortest the root tree id
    Also remove non-disease tree ids
    Args:
        - tree_ids (List): List of tree ids
    Returns:
        - List: List of shortened tree ids
    """
    return [x.split(".")[0] for x in tree_ids if x.startswith("C")]


def get_dsaid_relationships(
    disease_id: List,
    disease_tree: List,
    rest_ids: List,
    rest_trees: List,
    tree_root: str,
    relationship: str,
) -> np.array:
    """Get idxs for dsaids based on relationship
    Args:
        - disease_id (List): Disease ID
        - disease_tree (List): Disease Tree
        - rest_ids (List): Rest of the IDs
        - rest_trees (List): Rest of the Trees
        - tree_root (str): Root Tree
        - relationship (str): Relationship
    Returns:
        - idxs (np.array): Array of indexes
    """
    disease_tree = shorten_tree(disease_tree)
    rest_trees = shorten_tree(rest_trees)

    idxs = list()

    for idx in range(len(rest_ids)):
        rest_id_i = rest_ids[idx]
        rest_tree_i = rest_trees[idx]

        # disease id
        condition_1 = disease_id == rest_id_i

        # Related MeSH trees w/ disease
        condition_2 = len(set(disease_tree).intersection(set(rest_tree_i))) > 0

        # Same root tree
        condition_3 = len(set([tree_root]).intersection(set(rest_tree_i))) > 0

        if relationship == "same_disease":

            # Must have same Disease ID!
            if condition_1:
                idxs.append(idx)

        elif relationship == "related_disease":

            # Different Disease ID but same family as Root Tree
            if condition_1 and condition_3:
                idxs.append(idx)

        elif relationship == "unrelated_disease":

            # Different Disease ID and different family to Disease at all levels!
            if condition_1 and not condition_2:
                idxs.append(idx)

    return np.array(idxs)


def parse_mesh_data(file_path):
    """Parse MeSH XML data and extract disease terms."""
    tree = ET.parse(file_path)
    root = tree.getroot()

    # Extract disease terms
    disease_terms = list()
    list_tree_numbers = list()
    for descriptor in root.findall("DescriptorRecord"):
        # Check if the term is under the category of diseases
        tree_numbers = descriptor.findall("TreeNumberList/TreeNumber")
        for tree_number in tree_numbers:
            # This is a basic check for TreeNumbers starting with 'C' which usually denotes diseases in MeSH
            # You might need to adjust this based on the specific structure of your XML file
            if tree_number.text.startswith("C"):
                list_tree_numbers.append(tree_number.text)
                term = descriptor.find("DescriptorName/String").text
                disease_terms.append(term)
                break  # Break after adding the term to avoid duplicates

    return disease_terms, list_tree_numbers


def parse_mesh_data(file_path):
    """Parse MeSH XML data and extract disease terms."""
    tree = ET.parse(file_path)
    root = tree.getroot()

    # Dictionary to map tree numbers to disease terms
    tree_2_term = {}
    term_2_tree = {}

    # Extract disease terms and tree numbers
    for descriptor in root.findall("DescriptorRecord"):
        # Get the disease term name
        term = descriptor.find("DescriptorName/String").text

        # Get all tree numbers for this term
        tree_numbers = descriptor.findall("TreeNumberList/TreeNumber")

        for tree_number in tree_numbers:
            # Check for disease-related tree numbers starting with 'C'
            # if tree_number.text.startswith("C"):
            # Map each tree number to its term
            tree_2_term[tree_number.text] = term
            term_2_tree[term] = tree_number.text

    return tree_2_term, term_2_tree


# 2. Load Data
(
    # split,
    # predictions_test,
    labels_test,
    # results_test,
    # all_outputs_test,
    # predictions_train,
    labels_train,
    # results_train,
    # all_outputs_train,
    adata_orig,
    # id2type,
) = load_run_output(run_dir)

# load json

with open(os.path.join(run_dir, "parameters.json"), "r") as f:
    parameters = json.load(f)

for k, v in parameters.items():
    print(f"{k}: {v}")


# define train & test splits
split_n = 0
adata_test = adata_orig.obs[adata_orig.obs[f"test_split_{split_n+1}"] == 1]
adata_train = adata_orig[adata_orig.obs[f"test_split_{split_n+1}"] == 0]


# load DiSignAtlas info
df_info = pd.read_csv(df_info_path)

# load MeSH data
# disease_terms, list_tree_numbers = parse_mesh_data(file_path=mesh_file_path)
# print(disease_terms)

# tree_2_term = dict(zip(list_tree_numbers, disease_terms))
# term_2_tree = dict(zip(disease_terms, list_tree_numbers))

tree_2_term, term_2_tree = parse_mesh_data(file_path=mesh_file_path)

# map DisignAtlas to MeSH tree
mesh_da_mappings = pickle.load(open(mesh_da_mappings_path, "rb"))

# map samples to mesh terms
adata_orig.obs["dsaid"]

# get disease terms
disease_terms = adata_orig.obs["disease_study"].to_list()

Nº of loaded variables 3
data_path: /aloy/home/ddalton/projects/scGPT_playground/data/pp_data-24-10-06-01/data.h5ad
max_seq_len: 3501
batch_size: 16
gene_presence_pct: 0.9
benchmark_data: False
split_type: stratified
n_splits: 3
n_tested_splits: 3
epochs: 20
gene_filtering: top_presence
sample_presence_pct: 0.3


In [5]:
#! Filter out diseases non-disease root terms

# map dsaids & meshids
dsaid_2_meshid = dict()
root_2_mesh_id = dict()
mesh_id_2_tree_terms = dict()
for i in range(len(mesh_da_mappings["dsaids"])):
    if len(mesh_da_mappings["mesh_ids"][i]) > 0:
        dsaid_2_meshid[mesh_da_mappings["dsaids"][i]] = mesh_da_mappings["mesh_ids"][i][
            0
        ]

        # loop through ROOT tree terms
        for root_term in shorten_tree(mesh_da_mappings["mesh_tree_terms"][i]):
            if root_term not in root_2_mesh_id.keys():
                root_2_mesh_id[root_term] = set()
            root_2_mesh_id[root_term].add(mesh_da_mappings["mesh_ids"][i][0])

        # map mesh id to tree terms
        mesh_id_2_tree_terms[mesh_da_mappings["mesh_ids"][i][0]] = mesh_da_mappings[
            "mesh_tree_terms"
        ][i]
    else:
        dsaid_2_meshid[mesh_da_mappings["dsaids"][i]] = None

In [31]:
from collections import Counter
from typing import *
from tqdm import tqdm


def get_relationship(
    mesh_id: str, root_id: str, df_obs: pd.DataFrame, mesh_id_2_tree_terms: dict
) -> List:
    """Get Relationship between two diseases
    Args:
        - mesh_id (str): MeSH ID
        - root_id (str): Root ID
        - df_obs (pd.DataFrame): DataFrame with disease information
        - mesh_id_2_tree_terms (dict): Mapping of MeSH ID to Tree Terms
    Returns:
        - relationship (List): List of relationships
    """

    # get root tree terms
    tree_term = shorten_tree(mesh_id_2_tree_terms[mesh_id])

    relationship = list()

    for i, row in df_obs.iterrows():
        disease_i = row["disease"]
        mesh_id_i = row["mesh_id"]
        tree_term_i = shorten_tree(mesh_id_2_tree_terms[mesh_id_i])

        condition_1 = mesh_id == mesh_id_i
        condition_2 = disease_i == "Control"
        condition_3 = len(set([root_id]).intersection(set(tree_term_i))) > 0
        condition_4 = len(set(tree_term).intersection(set(tree_term_i))) > 0

        i = 0
        # Same Disease & not control
        if condition_1 and not condition_2:
            i += 1
            relationship.append("same disease")

        # Same Disease & control
        elif condition_1 and condition_2:
            i += 1
            relationship.append("control")

        # Different Disease & not control
        elif condition_3 and not condition_2:
            i += 1
            relationship.append("related disease")

        # Different Disease & not control
        elif condition_4 and not condition_2:
            i += 1
            relationship.append("related disease - different root")

        # Different Disease & control
        elif condition_4 and condition_2:
            i += 1
            relationship.append("related disease control")

        # Unrelated disease
        elif not condition_4 and not condition_2:
            i += 1
            relationship.append("unrelated disease")

        # Unrelated control
        elif not condition_4 and condition_2:
            i += 1
            relationship.append("unrelated control")

    return relationship


def parse_mesh_data(file_path):
    """Parse MeSH XML data and extract disease terms."""
    tree = ET.parse(file_path)
    root = tree.getroot()

    # Dictionary to map tree numbers to disease terms
    tree_2_term = {}
    term_2_tree = {}

    # Extract disease terms and tree numbers
    for descriptor in root.findall("DescriptorRecord"):
        # Get the disease term name
        term = descriptor.find("DescriptorName/String").text

        # Get all tree numbers for this term
        tree_numbers = descriptor.findall("TreeNumberList/TreeNumber")

        for tree_number in tree_numbers:
            # Check for disease-related tree numbers starting with 'C'
            # if tree_number.text.startswith("C"):
            # Map each tree number to its term
            tree_2_term[tree_number.text] = term
            term_2_tree[term] = tree_number.text

    return tree_2_term, term_2_tree


def parse_mesh_data_with_ids(file_path):
    """Parse MeSH XML data and extract disease terms along with MeSH IDs."""
    tree = ET.parse(file_path)
    root = tree.getroot()

    # Dictionaries to map tree numbers and MeSH IDs to disease terms
    tree_2_term = {}
    term_2_tree = {}
    mesh_id_2_term = {}

    # Extract disease terms, tree numbers, and MeSH IDs
    for descriptor in root.findall("DescriptorRecord"):
        # Get the disease term name
        term = descriptor.find("DescriptorName/String").text

        # Get the MeSH ID
        mesh_id = descriptor.find("DescriptorUI").text
        mesh_id_2_term[mesh_id] = term

        # Get all tree numbers for this term
        tree_numbers = descriptor.findall("TreeNumberList/TreeNumber")

        for tree_number in tree_numbers:
            # Map each tree number to its term
            tree_2_term[tree_number.text] = term
            term_2_tree[term] = tree_number.text

    return tree_2_term, term_2_tree, mesh_id_2_term


mesh_file_path = os.path.join(
    "/aloy/home/ddalton/projects/disease_signatures/data/MeSH/desc2023.xml"
)
tree_2_term, term_2_tree, mesh_id_2_term = parse_mesh_data_with_ids(
    file_path=mesh_file_path
)

In [34]:
from itertools import product
from tqdm import tqdm
from itertools import permutations

split_n = 0
split_type = "train"

# duplicate adata obs
df_obs = adata_orig.obs.copy()

# filter df_obs by test split
if split_type == "test":
    df_obs = df_obs[df_obs[f"test_split_{split_n+1}"] == 1]
elif split_type == "train":
    df_obs = df_obs[df_obs[f"test_split_{split_n+1}"] == 0]

# add mesh ids to obs
df_obs["mesh_id"] = df_obs["dsaid"].map(dsaid_2_meshid)

# reset indexes
df_obs.reset_index(inplace=True)


# index pairs
d_indexes = {
    "same_dis_dt": [],
    "control": [],
    "same_dis": [],
    "related_dis": [],
    "unrelated": [],
    "root": [],
    "mesh_id": [],
}

root_terms = [x for x in root_2_mesh_id.keys() if x.startswith("C")]
# root_terms = ["C04"]
for root_i in root_terms:
    # variables for each disease
    same_dis_dt_i = list()
    control_i = list()
    same_dis_i = list()
    related_dis_i = list()
    unrelated_i = list()

    # mesh ids present in the dataset
    mesh_ids = list(set(root_2_mesh_id[root_i]) & set(df_obs["mesh_id"].unique()))

    for mesh_id_j in tqdm(mesh_ids):
        # variables for each disease
        same_dis_dt_j = list()
        control_j = list()
        same_dis_j = list()
        related_dis_j = list()
        unrelated_j = list()

        # get relationship
        df_obs[f"relationship"] = get_relationship(
            mesh_id_j, root_i, df_obs, mesh_id_2_tree_terms
        )

        # query same disease relationships
        QUERY = "relationship == 'same disease'"
        df_query = df_obs.query(QUERY)
        for dataset_k in df_query["dataset"].unique():
            # get same disease indexes
            QUERY = "dataset == @dataset_k and relationship == 'same disease'"
            same_dis_st_idxs = df_obs.query(QUERY).index.to_list()

            # get control indexes
            QUERY = "dataset == @dataset_k and relationship == 'control'"
            control_idxs = df_obs.query(QUERY).index.to_list()

            # get same disease indexes
            QUERY = "dataset != @dataset_k and relationship == 'same disease'"
            same_dis_idxs = df_obs.query(QUERY).index.to_list()

            # get related disease indexes
            QUERY = "relationship == 'related disease'"
            related_dis_idxs = df_obs.query(QUERY).index.to_list()

            # get unrelated disease indexes
            QUERY = "relationship == 'unrelated disease'"
            unrelated_idxs = df_obs.query(QUERY).index.to_list()

            # compute index pairs
            same_dis_dt_j.extend(list(permutations(same_dis_st_idxs, 2)))
            control_j.extend(list(product(same_dis_st_idxs, control_idxs)))
            same_dis_j.extend(list(product(same_dis_st_idxs, same_dis_idxs)))
            related_dis_j.extend(list(product(same_dis_st_idxs, related_dis_idxs)))
            unrelated_j.extend(list(product(same_dis_st_idxs, unrelated_idxs)))
        # store results
        same_dis_dt_i.append(np.array(same_dis_dt_j, dtype=int))
        control_i.append(np.array(control_j, dtype=int))
        same_dis_i.append(np.array(same_dis_j, dtype=int))
        related_dis_i.append(np.array(related_dis_j, dtype=int))
        unrelated_i.append(np.array(unrelated_j, dtype=int))

    # store results
    d_indexes["same_dis_dt"].append(same_dis_dt_i)
    d_indexes["control"].append(control_i)
    d_indexes["same_dis"].append(same_dis_i)
    d_indexes["related_dis"].append(related_dis_i)
    d_indexes["unrelated"].append(unrelated_i)
    d_indexes["root"].append(root_i)
    d_indexes["mesh_id"].append(mesh_ids)

100%|██████████| 1/1 [00:02<00:00,  2.60s/it]
0it [00:00, ?it/s]
100%|██████████| 1/1 [00:01<00:00,  1.65s/it]
0it [00:00, ?it/s]


In [16]:
# for root_i in root_2_mesh_id.keys():
#     # mesh ids present in the dataset
#     mesh_ids = list(set(root_2_mesh_id[root_i]) & set(df_obs["mesh_id"].unique()))
#     if len(mesh_ids) > 2:
#         print(f"Root Tree: {len(mesh_ids)} - {tree_2_term.get(root_i)}")

Root Tree: 9 - Nervous System Diseases
Root Tree: 8 - Mental Disorders
Root Tree: 24 - Neoplasms
Root Tree: 4 - Musculoskeletal Diseases
Root Tree: 9 - Congenital, Hereditary, and Neonatal Diseases and Abnormalities
Root Tree: 9 - Pathological Conditions, Signs and Symptoms
Root Tree: 8 - Cardiovascular Diseases
Root Tree: 14 - Digestive System Diseases
Root Tree: 8 - Hemic and Lymphatic Diseases
Root Tree: 14 - Immune System Diseases
Root Tree: 9 - Urogenital Diseases
Root Tree: 6 - Endocrine System Diseases
Root Tree: 6 - Nutritional and Metabolic Diseases
Root Tree: 7 - Respiratory Tract Diseases
Root Tree: 7 - Skin and Connective Tissue Diseases
Root Tree: 4 - Infections


In [25]:
comparison_matrix = load_comparison_matrix(output_dir, "Cosine Similarity", split_type)

In [37]:
def process_index_pairs(idx_pairs: List) -> Tuple[np.array, np.array]:
    """Process index pairs
    Args:
        - idx_pairs (List): List of index pairs
    Returns:
        - rows (np.array): Array of row indices
        - cols (np.array): Array of column indices
    """
    # Separate the pairs into row and column indices
    rows, cols = zip(*idx_pairs)  # Unzips into two lists

    # Convert to NumPy arrays for advanced indexing
    rows = np.array(rows)
    cols = np.array(cols)
    return rows, cols


def get_matrix_elements(comparison_matrix: np.array, idx_pairs: List) -> np.array:
    """Get matrix elements for a given set of index pairs
    Args:
        - comparison_matrix (np.array): Comparison matrix
        - idx_pairs (List): List of index pairs
    Returns:
        - elements (np.array): Array of matrix elements
    """

    # Separate the pairs into row and column indices
    rows, cols = process_index_pairs(idx_pairs)

    # Get the matrix elements
    elements = comparison_matrix[rows, cols]

    return elements


for i, root in enumerate(d_indexes["root"]):
    print(f"Root: {root}")
    for j, mesh_id in enumerate(d_indexes["mesh_id"][i]):
        same_dis_dt_i_j = d_indexes["same_dis_dt"][i][j]
        control_i_j = d_indexes["control"][i][j]
        same_dis_i_j = d_indexes["same_dis"][i][j]
        related_dis_i_j = d_indexes["related_dis"][i][j]
        unrelated_i_j = d_indexes["unrelated"][i][j]

        if np.all(
            [
                True if len(x) > 0 else False
                for x in [
                    same_dis_dt_i_j,
                    control_i_j,
                    same_dis_i_j,
                    related_dis_i_j,
                    unrelated_i_j,
                ]
            ]
        ):

            same_dis_dt_v = get_matrix_elements(comparison_matrix, same_dis_dt_i_j)
            control_v = get_matrix_elements(comparison_matrix, control_i_j)
            same_dis_v = get_matrix_elements(comparison_matrix, same_dis_i_j)
            related_dis_v = get_matrix_elements(comparison_matrix, related_dis_i_j)
            unrelated_v = get_matrix_elements(comparison_matrix, unrelated_i_j)

            plt.figure(figsize=(5, 5))

            plt.title(
                f"Root: {tree_2_term.get(root)}\nMesh ID: {mesh_id_2_term.get(mesh_id)}"
            )

            sns.kdeplot(same_dis_dt_v, label="Same Disease Same Dataset")
            sns.kdeplot(control_v, label="Control")
            sns.kdeplot(same_dis_v, label="Same Disease Different Dataset")
            sns.kdeplot(related_dis_v, label="Related Disease")
            sns.kdeplot(unrelated_v, label="Unrelated Disease")
            plt.legend()
            plt.savefig(
                os.path.join(
                    output_dir,
                    f"{tree_2_term.get(root)}_{mesh_id_2_term.get(mesh_id)}.png",
                ),
                dpi=300,
                bbox_inches="tight",
            )
            plt.close()

Root: C10
Root: C04
Root: C11
Root: C05
Root: C16
Root: C23
Root: C14
Root: C06
Root: C15
Root: C20
Root: C12
Root: C19
Root: C18
Root: C08
Root: C17
Root: C01
Root: C07
Root: C09
Root: C26
Root: C25
Root: C22


In [29]:
np.all(
    [
        True if len(x) > 0 else False
        for x in [same_dis_dt_i_j, control_i_j, related_dis_i_j, unrelated_i_j]
    ]
)

True

In [16]:
def get_grouped_ps(matrix: np.array, idxs: np.array, labels: List) -> Dict:
    """Get Grouped Pairwise Similarities
    Args:
        matrix (np.array): Pairwise similarity matrix
        idxs (List): List of indexes
        labels (List): List of labels
    Returns:
        grouped_ps (Dict): Dictionary with grouped pairwise similarities
    """
    grouped_ps = dict()
    label_idxs = list(range(len(labels)))
    for i, idx_i in enumerate(label_idxs):
        for idx_j in label_idxs[i:]:
            label_i = labels[idx_i]
            label_j = labels[idx_j]

            if idx_i == idx_j:
                grouped_ps[f"{label_i}-{label_j}"] = get_pairwise_similarities(
                    matrix, idxs[idx_i]
                )
            else:
                grouped_ps[f"{label_i}-{label_j}"] = get_pairwise_similarities(
                    matrix, idxs[idx_i], idxs[idx_j]
                )
    return grouped_ps

In [ ]:
metric_params = {
    "Cosine Similarity": (-1, 1),
    "Euclidean Distance": (None, None),  # Empty tuple if you don't want to set bounds
    "Pearson Correlation": (-1, 1),
}


# for data_type in ["train", "test"]:
for data_type in ["train"]:

    if data_type == "train":
        adata = adata_train
        c_matrix = c_matrix_train
        # e_matrix = e_matrix_train
        # p_matrix = p_matrix_train
    else:
        adata = adata_test
        c_matrix = c_matrix_test
        # e_matrix = e_matrix_test
        # p_matrix = p_matrix_test

    # Compute pairwise similarities for each disease
    for disease_it in adata.obs["disease_study"].unique()[:10]:

        # get disease & control indexes
        disease_idxs = list()
        control_idxs = list()
        other_disease_idxs = list()
        for i, (disease, disease_study) in enumerate(
            zip(adata.obs["disease"].to_list(), adata.obs["disease_study"].to_list())
        ):
            if (disease == disease_it) & (disease_study == disease_it):
                disease_idxs.append(i)
            elif (disease == "Control") & (disease_study == disease_it):
                control_idxs.append(i)
            elif disease != disease_study:
                other_disease_idxs.append(i)

        print(f"Control: {len(control_idxs)}")
        print(f"Disease: {len(disease_idxs)}")
        n_samples = len(disease_idxs)
        # Plot Metrics
        for metric, matrix in zip(
            [
                "Cosine Similarity",
            ],
            [c_matrix],
        ):

            d_ps = get_grouped_ps(
                matrix,
                [disease_idxs, control_idxs, other_disease_idxs],
                ["Disease", "Control", "Other Disease"],
            )

            d_data = {k: v for k, v in d_ps.items() if k.startswith("Disease-")}

            plt.figure(figsize=(4, 4))

            sns.kdeplot(
                d_data["Disease-Other Disease"],
                label="Disease-Other Disease",
                fill=True,
                alpha=0.8,
            )
            sns.kdeplot(
                d_data["Disease-Control"], label="Disease-Control", fill=True, alpha=0.5
            )
            sns.kdeplot(
                d_data["Disease-Disease"], label="Disease-Disease", fill=True, alpha=0.3
            )
            plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
            plt.title(f"{metric}\n{disease_it} {data_type} - ({n_samples})")
            plt.xlim(metric_params[metric])

            plt.show()

            # only one metric

In [ ]:
adata_train

In [ ]:
adata_train.obs["disease_study"].unique()[:10]

In [ ]:
"""Compute distances for same disease tree?
    - Same disease
    - Control
    - Related disease
    - Unrelated disease
"""

In [12]:
# load DiSignAtlas info
df_info = pd.read_csv(df_info_path)

# load MeSH data
# disease_terms, list_tree_numbers = parse_mesh_data(file_path=mesh_file_path)
# print(disease_terms)

# tree_2_term = dict(zip(list_tree_numbers, disease_terms))
# term_2_tree = dict(zip(disease_terms, list_tree_numbers))

tree_2_term, term_2_tree = parse_mesh_data(file_path=mesh_file_path)

# map DisignAtlas to MeSH tree
mesh_da_mappings = pickle.load(open(mesh_da_mappings_path, "rb"))

# map samples to mesh terms
adata_orig.obs["dsaid"]

# get disease terms
disease_terms = adata_orig.obs["disease_study"].to_list()

In [14]:
# get DA tree numbers
dsaid_2_tree = dict(
    zip(mesh_da_mappings["dsaids"], mesh_da_mappings["mesh_tree_terms"])
)
da_tree_numbers = [dsaid_2_tree.get(x) for x in adata_orig.obs["dsaid"].to_list()]

In [ ]:
from typing import *

# First retrieve which dsaids


# Control

# same disease different dataset


# different disease same dataset

In [ ]:
set(["as"])

In [ ]:
condition = True
condition_2 = False

if condition and not condition_2:
    print("True")